# Classification Spam / Non-Spam — Emails (Enron Spam Dataset)

**Auteur : Dimitri Mintsa**

Ce notebook entraîne un classifieur **spam / non-spam** sur de vrais emails, à partir du
dataset **Enron-Spam** (`SetFit/enron_spam`) — l'un des corpus email les plus utilisés en
classification de spam (format académique, texte complet sujet+corps). Approche **classique
robuste** : TF-IDF + modèles linéaires (Naive Bayes, Logistic Regression, LinearSVC),
comparés par validation croisée, puis le meilleur est exporté pour une intégration dans
un add-in Outlook.

> **Réglages Kaggle :** Internet = On. Pas besoin de GPU — ce type de modèle (TF-IDF +
> modèle linéaire) tourne très bien sur CPU, et c'est justement ce qui le rend adapté à un
> add-in local qui tourne en temps réel à chaque mail reçu.

## 1. Installation

In [ ]:
# ─── 1. Installation ─────────────────────────────────────────────────────────
import subprocess, sys
for pkg in ["datasets", "gradio", "wordcloud"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])
print("✅ Installation terminée")

## 2. Imports et configuration

In [ ]:
# ─── 2. Imports ──────────────────────────────────────────────────────────────
import os, re, json, random, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import joblib
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, classification_report,
                              RocCurveDisplay, PrecisionRecallDisplay)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
print("✅ Environnement prêt — TF-IDF + modèle linéaire : pas besoin de GPU, le CPU suffit.")

In [ ]:
# ─── 2b. Configuration ───────────────────────────────────────────────────────
DATASET_NAME = "SetFit/enron_spam"   # emails réels, labellisés spam/ham (standard académique)

OUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

TEST_SIZE = 0.2
MAX_FEATURES_TFIDF = 30000
NGRAM_RANGE = (1, 2)

print("Dossier de sortie :", OUT_DIR)

## 3. Téléchargement du dataset et statistiques

In [ ]:
# ─── 3. Téléchargement ───────────────────────────────────────────────────────
from datasets import load_dataset

raw = load_dataset(DATASET_NAME)
print(raw)

df_train = raw["train"].to_pandas()
df_test  = raw["test"].to_pandas()
df = pd.concat([df_train, df_test], ignore_index=True)

df["text"] = df["text"].astype(str)
print(f"Total emails : {len(df)}")
df.head(3)

In [ ]:
# ─── 3b. Statistiques ────────────────────────────────────────────────────────
print("Répartition des classes :")
print(df["label_text"].value_counts())

n_dup = df.duplicated(subset="text").sum()
print(f"\nDoublons exacts : {n_dup}")
df = df.drop_duplicates(subset="text").reset_index(drop=True)
print(f"Après dédoublonnage : {len(df)} emails")

df["n_chars"] = df["text"].str.len()
df["n_words"] = df["text"].str.split().apply(len)
print(f"\nLongueur (mots) — moy {df['n_words'].mean():.0f} | min {df['n_words'].min()} | max {df['n_words'].max()}")
print(f"Longueur (car)  — moy {df['n_chars'].mean():.0f} | min {df['n_chars'].min()} | max {df['n_chars'].max()}")

## 4. Visualisation

In [ ]:
# ─── 4. Distributions ────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

df["label_text"].value_counts().plot(kind="bar", ax=ax[0], color=["#4C6EF5", "#C17C2A"])
ax[0].set_title("Répartition des classes"); ax[0].set_xlabel("")

ax[1].hist(df[df.label_text == "ham"]["n_words"].clip(upper=500), bins=40, alpha=.6, label="ham", color="#4C6EF5")
ax[1].hist(df[df.label_text == "spam"]["n_words"].clip(upper=500), bins=40, alpha=.6, label="spam", color="#C17C2A")
ax[1].set_title("Longueur (mots, plafonné à 500)"); ax[1].legend()

ax[2].boxplot([df[df.label_text == "ham"]["n_words"].clip(upper=1000),
               df[df.label_text == "spam"]["n_words"].clip(upper=1000)], labels=["ham", "spam"])
ax[2].set_title("Longueur — boxplot")
plt.tight_layout(); plt.show()

In [ ]:
# ─── 4b. Nuages de mots par classe ───────────────────────────────────────────
def clean_for_wc(texts):
    txt = " ".join(texts).lower()
    txt = re.sub(r"[^a-zàâçéèêëîïôûùü\s]", " ", txt)
    return txt

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for i, lab in enumerate(["ham", "spam"]):
    pool = df[df.label_text == lab]["text"]
    sample_txt = clean_for_wc(pool.sample(min(1500, len(pool)), random_state=SEED))
    wc = WordCloud(width=700, height=500, background_color="white", max_words=100).generate(sample_txt)
    ax[i].imshow(wc); ax[i].axis("off"); ax[i].set_title(f"Mots fréquents — {lab}")
plt.tight_layout(); plt.show()

## 5. Prétraitement du texte

In [ ]:
# ─── 5. Nettoyage ─────────────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " url ", text)
    text = re.sub(r"\S+@\S+", " email ", text)
    text = re.sub(r"\d+", " num ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text_clean"] = df["text"].apply(clean_text)
df[["text", "text_clean"]].sample(3, random_state=SEED)

## 6. Split train / test

In [ ]:
# ─── 6. Split ─────────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    df["text_clean"], df["label"], test_size=TEST_SIZE, stratify=df["label"], random_state=SEED
)
print(f"Train : {len(X_train)} | Test : {len(X_test)}")
print("Répartition train :", y_train.value_counts(normalize=True).round(3).to_dict())
print("Répartition test  :", y_test.value_counts(normalize=True).round(3).to_dict())

## 7. Vectorisation TF-IDF

In [ ]:
# ─── 7. TF-IDF ────────────────────────────────────────────────────────────────
vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES_TFIDF,
    ngram_range=NGRAM_RANGE,
    sublinear_tf=True,
    stop_words="english",
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)
print(f"Vocabulaire : {len(vectorizer.vocabulary_)} tokens")
print(f"Matrice train : {X_train_tfidf.shape} | test : {X_test_tfidf.shape}")

## 8. Entraînement et comparaison des modèles candidats

In [ ]:
# ─── 8. Comparaison par validation croisée ───────────────────────────────────
candidates = {
    "MultinomialNB": MultinomialNB(),
    "LogisticRegression": LogisticRegression(max_iter=1000, C=10, random_state=SEED),
    "LinearSVC": LinearSVC(C=1.0, random_state=SEED),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = []
for name, model in candidates.items():
    scores = cross_val_score(model, X_train_tfidf, y_train, cv=cv, scoring="f1")
    results.append({"Modèle": name, "F1 (cv moy)": scores.mean(), "F1 (cv std)": scores.std()})
    print(f"{name:20s} F1 cv = {scores.mean():.4f} (+/- {scores.std():.4f})")

cv_results = pd.DataFrame(results).sort_values("F1 (cv moy)", ascending=False)
cv_results

## 9. Entraînement final + évaluation détaillée du meilleur modèle

In [ ]:
# ─── 9. Modèle final ──────────────────────────────────────────────────────────
BEST_NAME = cv_results.iloc[0]["Modèle"]
print("Modèle retenu :", BEST_NAME)

best_model = candidates[BEST_NAME]
if BEST_NAME == "LinearSVC":
    best_model = CalibratedClassifierCV(best_model)  # pour obtenir predict_proba
best_model.fit(X_train_tfidf, y_train)

y_pred = best_model.predict(X_test_tfidf)
y_proba = best_model.predict_proba(X_test_tfidf)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_proba),
}
for k, v in metrics.items():
    print(f"{k:10s}: {v:.4f}")

print("\n" + classification_report(y_test, y_pred, target_names=["ham", "spam"]))

In [ ]:
# ─── 9b. Matrice de confusion + ROC + Precision-Recall ──────────────────────
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

cm = confusion_matrix(y_test, y_pred)
ax[0].imshow(cm, cmap="Blues")
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(["ham", "spam"])
ax[0].set_yticks([0, 1]); ax[0].set_yticklabels(["ham", "spam"])
for i in range(2):
    for j in range(2):
        ax[0].text(j, i, cm[i, j], ha="center", va="center", color="black")
ax[0].set_title("Matrice de confusion"); ax[0].set_xlabel("Prédit"); ax[0].set_ylabel("Réel")

RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax[1], color="#4C6EF5")
ax[1].set_title("Courbe ROC")

PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=ax[2], color="#C17C2A")
ax[2].set_title("Courbe Precision-Recall")
plt.tight_layout(); plt.show()

## 10. Analyse des erreurs

In [ ]:
# ─── 10. Faux positifs / faux négatifs ───────────────────────────────────────
test_df = pd.DataFrame({
    "text": X_test.values, "label": y_test.values, "pred": y_pred, "proba_spam": y_proba
})
false_positives = test_df[(test_df.label == 0) & (test_df.pred == 1)].sort_values("proba_spam", ascending=False)
false_negatives = test_df[(test_df.label == 1) & (test_df.pred == 0)].sort_values("proba_spam")

print(f"Faux positifs (ham classé spam) : {len(false_positives)}")
print(f"Faux négatifs (spam classé ham) : {len(false_negatives)}")

print("\n--- Exemple faux positif ---")
if len(false_positives):
    print(false_positives.iloc[0]["text"][:400])

print("\n--- Exemple faux négatif ---")
if len(false_negatives):
    print(false_negatives.iloc[0]["text"][:400])

## 11. Récapitulatif

In [ ]:
# ─── 11. Récapitulatif ────────────────────────────────────────────────────────
recap = pd.DataFrame({
    "Paramètre": ["Dataset", "Emails (total)", "Vocabulaire TF-IDF", "N-grammes",
                  "Modèle retenu", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    "Valeur": [DATASET_NAME, len(df), len(vectorizer.vocabulary_), str(NGRAM_RANGE),
               BEST_NAME, f"{metrics['Accuracy']:.4f}", f"{metrics['Precision']:.4f}",
               f"{metrics['Recall']:.4f}", f"{metrics['F1']:.4f}", f"{metrics['ROC-AUC']:.4f}"],
})
display(recap)

## 12. Export des artefacts (modèle + vectorizer)

In [ ]:
# ─── 12. Export ───────────────────────────────────────────────────────────────
MODEL_DIR = f"{OUT_DIR}/model"
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(best_model, f"{MODEL_DIR}/spam_classifier.joblib")
joblib.dump(vectorizer, f"{MODEL_DIR}/tfidf_vectorizer.joblib")

import sklearn
meta = {
    "model_name": BEST_NAME,
    "dataset": DATASET_NAME,
    "metrics": metrics,
    "label_map": {"0": "ham", "1": "spam"},
    "sklearn_version": sklearn.__version__,
}
json.dump(meta, open(f"{MODEL_DIR}/metadata.json", "w"), indent=2)

print("✅ Artefacts exportés dans", MODEL_DIR)
for p in os.listdir(MODEL_DIR):
    full = f"{MODEL_DIR}/{p}"
    print(" •", p, f"({os.path.getsize(full)/1e3:.1f} Ko)")

## 13. Test Gradio

In [ ]:
# ─── 13. Interface Gradio de test ─────────────────────────────────────────────
import subprocess, sys
try:
    import gradio as gr
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"])
    import gradio as gr
print("gradio", gr.__version__)

clf = joblib.load(f"{MODEL_DIR}/spam_classifier.joblib")
vec = joblib.load(f"{MODEL_DIR}/tfidf_vectorizer.joblib")

def predict_spam(email_text):
    if not email_text or not email_text.strip():
        return "—", 0.0
    cleaned = clean_text(email_text)
    X = vec.transform([cleaned])
    proba = clf.predict_proba(X)[0, 1]
    label = "🚫 SPAM" if proba >= 0.5 else "✅ NON-SPAM"
    return label, round(float(proba), 4)

with gr.Blocks(title="Détecteur de spam — TF-IDF + " + BEST_NAME) as demo:
    gr.Markdown(f"## 📧 Détecteur de spam email ({BEST_NAME})")
    with gr.Row():
        with gr.Column():
            email_in = gr.Textbox(label="Texte de l'email (sujet + corps)", lines=8,
                                  placeholder="Colle ici le contenu d'un email...")
            btn = gr.Button("Analyser 🔍", variant="primary")
        with gr.Column():
            label_out = gr.Textbox(label="Résultat")
            proba_out = gr.Number(label="Probabilité spam")
    btn.click(predict_spam, [email_in], [label_out, proba_out])
    gr.Examples([
        ["CONGRATULATIONS! You've WON a $1,000,000 prize! Click here now to claim your reward before it expires!!!"],
        ["Hi team, please find attached the quarterly report. Let me know if you have any questions before the meeting tomorrow."],
    ], inputs=[email_in])

demo.launch(share=True, debug=True)